# HiRAG Retrieval q50 Case Studies

This notebook inspects individual queries from the q50 retrieval benchmark.

The aggregate notebook shows that `hi_minmax` improves bridge coverage most, but also increases bridge size. Here we look at concrete queries to understand where the strategies differ.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 80)

RUN_DIR_CANDIDATES = [
    Path(".runs/retrieval_eval/dev_retrieval_q50"),
    Path("../.runs/retrieval_eval/dev_retrieval_q50"),
]
RUN_DIR = next((path for path in RUN_DIR_CANDIDATES if path.exists()), RUN_DIR_CANDIDATES[0])
metrics = pd.read_csv(RUN_DIR / "metrics.csv")

contexts = []
with (RUN_DIR / "contexts.jsonl").open("r", encoding="utf-8") as handle:
    for line in handle:
        contexts.append(json.loads(line))
contexts = pd.DataFrame(contexts)

variant_order = ["hi", "hi_weighted", "hi_minmax", "hi_rerank", "hi_rerank_weighted"]
metrics.shape, contexts.shape

## Find Query-Level Wins And Losses

A win here means a variant has higher bridge/query token overlap than baseline `hi` for the same query. This is only a proxy for answer quality, but it helps us find useful examples to read.

In [ ]:
base = metrics[metrics.variant == "hi"].set_index("query")

def variant_delta(variant, metric="bridge_vs_query_overlap"):
    rows = metrics[metrics.variant == variant].set_index("query")
    joined = rows.join(base, lsuffix="", rsuffix="_hi")
    joined["delta"] = joined[metric] - joined[f"{metric}_hi"]
    return joined.reset_index().sort_values("delta", ascending=False)

minmax_query_overlap = variant_delta("hi_minmax", "bridge_vs_query_overlap")
minmax_query_overlap[["query", "delta", "bridge_vs_query_overlap", "bridge_vs_query_overlap_hi", "context_tokens", "context_tokens_hi", "bridge_path_edges", "bridge_path_edges_hi"]].head(10)

In [ ]:
minmax_query_overlap[["query", "delta", "bridge_vs_query_overlap", "bridge_vs_query_overlap_hi", "context_tokens", "context_tokens_hi", "bridge_path_edges", "bridge_path_edges_hi"]].tail(10)

## Helper Functions For Reading A Query

The benchmark stores full contexts, selected entities, communities, and bridge edges. The helpers below print a compact view instead of dumping the whole 30k-token context.

In [ ]:
def get_context(query, variant):
    row = contexts[(contexts["query"] == query) & (contexts["variant"] == variant)]
    if row.empty:
        raise KeyError((query, variant))
    return row.iloc[0].to_dict()

def get_metrics(query, variant):
    row = metrics[(metrics["query"] == query) & (metrics["variant"] == variant)]
    if row.empty:
        raise KeyError((query, variant))
    return row.iloc[0].to_dict()

def compact_edges(edges, n=12):
    rows = []
    for edge in edges[:n]:
        rows.append({
            "source": edge.get("source"),
            "target": edge.get("target"),
            "rank": edge.get("rank"),
            "semantic_score": edge.get("semantic_score"),
            "cost": edge.get("cost"),
            "description": edge.get("description", "")[:180],
        })
    return pd.DataFrame(rows)

def compare_query(query, variants=("hi", "hi_minmax", "hi_rerank_weighted")):
    metric_rows = []
    for variant in variants:
        m = get_metrics(query, variant)
        metric_rows.append({
            "variant": variant,
            "context_tokens": m["context_tokens"],
            "bridge_edges": m["bridge_edge_count"],
            "path_edges": m["bridge_path_edges"],
            "query_overlap": m["bridge_vs_query_overlap"],
            "local_recall": m["bridge_vs_local_recall"],
            "global_recall": m["bridge_vs_global_recall"],
            "latency": m["latency_seconds"],
        })
    return pd.DataFrame(metric_rows)

def selected_entities(query, variant, n=20):
    return pd.DataFrame({"entity": get_context(query, variant)["selected_entities"][:n]})

## Example 1: Strong Minmax Win

This query had one of the largest bridge/query-overlap improvements for `hi_minmax`.

In [ ]:
win_query = minmax_query_overlap.iloc[0]["query"]
win_query

In [ ]:
compare_query(win_query)

In [ ]:
pd.concat(
    {
        "hi": selected_entities(win_query, "hi"),
        "hi_minmax": selected_entities(win_query, "hi_minmax"),
        "hi_rerank_weighted": selected_entities(win_query, "hi_rerank_weighted"),
    },
    axis=1,
)

In [ ]:
compact_edges(get_context(win_query, "hi")["bridge_edges"], n=10)

In [ ]:
compact_edges(get_context(win_query, "hi_minmax")["bridge_edges"], n=10)

Reading prompt: compare whether minmax is adding genuinely query-relevant bridge edges or merely adding more path text. If it helps because it finds better intermediate concepts, that supports budgeted minmax as the next implementation.

## Example 2: Minmax Loss Or No-Gain Case

This query is useful because it shows where minmax does not improve query overlap. These are the cases that tell us what budget/penalty to add.

In [ ]:
loss_query = minmax_query_overlap.iloc[-1]["query"]
loss_query

In [ ]:
compare_query(loss_query)

In [ ]:
compact_edges(get_context(loss_query, "hi")["bridge_edges"], n=12)

In [ ]:
compact_edges(get_context(loss_query, "hi_minmax")["bridge_edges"], n=12)

## Reranking Changes The Seed Entities

The reranker variants often shorten the bridge, but reduce bridge/local recall. This section finds queries where `hi_rerank_weighted` changes bridge size the most.

In [ ]:
rerank_edges = variant_delta("hi_rerank_weighted", "bridge_edge_count")
rerank_edges[["query", "delta", "bridge_edge_count", "bridge_edge_count_hi", "bridge_vs_query_overlap", "bridge_vs_query_overlap_hi"]].head(8)

In [ ]:
rerank_edges[["query", "delta", "bridge_edge_count", "bridge_edge_count_hi", "bridge_vs_query_overlap", "bridge_vs_query_overlap_hi"]].tail(8)

In [ ]:
compact_query = rerank_edges.iloc[-1]["query"]
compact_query

In [ ]:
compare_query(compact_query, variants=("hi", "hi_rerank", "hi_rerank_weighted"))

In [ ]:
pd.concat(
    {
        "hi": selected_entities(compact_query, "hi"),
        "hi_rerank": selected_entities(compact_query, "hi_rerank"),
        "hi_rerank_weighted": selected_entities(compact_query, "hi_rerank_weighted"),
    },
    axis=1,
)

## Query-Level Heatmap

This heatmap shows per-query bridge/query-overlap deltas versus baseline. Blue means the variant improved the deterministic query-overlap proxy; red means it got worse.

In [ ]:
heat = []
for variant in variant_order[1:]:
    d = variant_delta(variant, "bridge_vs_query_overlap")
    d["short_query"] = d["query"].str.slice(0, 55)
    heat.append(d[["short_query", "variant", "delta"]])
heat = pd.concat(heat)
pivot = heat.pivot_table(index="short_query", columns="variant", values="delta", aggfunc="first")

plt.figure(figsize=(9, 14))
sns.heatmap(pivot, center=0, cmap="vlag", linewidths=0.2, cbar_kws={"label": "Delta vs hi"})
plt.title("Per-query bridge/query-overlap delta vs baseline")
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()
plt.show()

## How To Use This Notebook For The Report

Use the aggregate notebook for the main quantitative story, then use this notebook to choose 2-3 representative examples:

1. A minmax win where the path clearly picks better intermediate concepts.
2. A minmax no-gain/loss where the path gets long without helping.
3. A reranking case where the seed entities visibly change.

These examples make the project easier to explain than metrics alone.